In [1]:
import json
import sqlite3

import pandas as pd

conn = sqlite3.connect("../data/happiness_raw.db")
raw = pd.read_sql_query("SELECT * FROM raw_records", conn)

print(raw.shape)
print(raw.groupby("year").size())
raw.head()

(782, 7)
year
2015    158
2016    157
2017    155
2018    156
2019    156
dtype: int64


,record_id,year,source_file,kafka_partition,kafka_offset,kafka_ts,payload
0,2015-4,2015,2015.csv,2,0,1794769254624,"{""Country"": ""Norway"", ""Region"": ""Western Europ..."
1,2015-6,2015,2015.csv,2,1,1794769254625,"{""Country"": ""Finland"", ""Region"": ""Western Euro..."
2,2015-8,2015,2015.csv,2,2,1794769254625,"{""Country"": ""Sweden"", ""Region"": ""Western Europ..."
3,2015-9,2015,2015.csv,2,3,1794769254625,"{""Country"": ""New Zealand"", ""Region"": ""Australi..."
4,2015-12,2015,2015.csv,2,4,1794769254625,"{""Country"": ""Costa Rica"", ""Region"": ""Latin Ame..."


In [2]:
ejemplo = raw.iloc[0]
print("Metadatos:", {k: v for k, v in ejemplo.items() if k != "payload"})
print("\nCampos originales:")
print(json.dumps(json.loads(ejemplo["payload"]), indent=2))

Metadatos: {'record_id': '2015-4', 'year': np.int64(2015), 'source_file': '2015.csv', 'kafka_partition': np.int64(2), 'kafka_offset': np.int64(0), 'kafka_ts': np.int64(1794769254624)}

Campos originales:
{
  "Country": "Norway",
  "Region": "Western Europe",
  "Happiness Rank": 4,
  "Happiness Score": 7.522,
  "Standard Error": 0.0388,
  "Economy (GDP per Capita)": 1.459,
  "Family": 1.33095,
  "Health (Life Expectancy)": 0.88521,
  "Freedom": 0.66973,
  "Trust (Government Corruption)": 0.36503,
  "Generosity": 0.34699,
  "Dystopia Residual": 2.46531
}


In [3]:
por_anio = {}
for year, g in raw.groupby("year"):
    por_anio[year] = pd.DataFrame([json.loads(p) for p in g["payload"]])

for year, df in por_anio.items():
    print(f"--- {year}: {df.shape}")
    print(df.dtypes.to_string())
    print("nulos:", df.isna().sum()[lambda s: s > 0].to_dict())
    print()

--- 2015: (158, 12)
Country                              str
Region                               str
Happiness Rank                     int64
Happiness Score                  float64
Standard Error                   float64
Economy (GDP per Capita)         float64
Family                           float64
Health (Life Expectancy)         float64
Freedom                          float64
Trust (Government Corruption)    float64
Generosity                       float64
Dystopia Residual                float64
nulos: {}

--- 2016: (157, 13)
Country                              str
Region                               str
Happiness Rank                     int64
Happiness Score                  float64
Lower Confidence Interval        float64
Upper Confidence Interval        float64
Economy (GDP per Capita)         float64
Family                           float64
Health (Life Expectancy)         float64
Freedom                          float64
Trust (Government Corruption)    float64
Genero

In [4]:
cols = {y: set(df.columns) for y, df in por_anio.items()}
comunes = set.intersection(*cols.values())
print("Comunes a todos los años:", sorted(comunes))
for y, c in cols.items():
    print(y, "no comunes:", sorted(c - comunes))

presencia = (pd.DataFrame({y: pd.Series(1, index=list(c)) for y, c in cols.items()})
             .fillna(0).astype(int).sort_index())
presencia

Comunes a todos los años: ['Generosity']
2015 no comunes: ['Country', 'Dystopia Residual', 'Economy (GDP per Capita)', 'Family', 'Freedom', 'Happiness Rank', 'Happiness Score', 'Health (Life Expectancy)', 'Region', 'Standard Error', 'Trust (Government Corruption)']
2016 no comunes: ['Country', 'Dystopia Residual', 'Economy (GDP per Capita)', 'Family', 'Freedom', 'Happiness Rank', 'Happiness Score', 'Health (Life Expectancy)', 'Lower Confidence Interval', 'Region', 'Trust (Government Corruption)', 'Upper Confidence Interval']
2017 no comunes: ['Country', 'Dystopia.Residual', 'Economy..GDP.per.Capita.', 'Family', 'Freedom', 'Happiness.Rank', 'Happiness.Score', 'Health..Life.Expectancy.', 'Trust..Government.Corruption.', 'Whisker.high', 'Whisker.low']
2018 no comunes: ['Country or region', 'Freedom to make life choices', 'GDP per capita', 'Healthy life expectancy', 'Overall rank', 'Perceptions of corruption', 'Score', 'Social support']
2019 no comunes: ['Country or region', 'Freedom to ma

,2015,2016,2017,2018,2019
Country,1,1,1,0,0
Country or region,0,0,0,1,1
Dystopia Residual,1,1,0,0,0
Dystopia.Residual,0,0,1,0,0
Economy (GDP per Capita),1,1,0,0,0
Economy..GDP.per.Capita.,0,0,1,0,0
Family,1,1,1,0,0
Freedom,1,1,1,0,0
Freedom to make life choices,0,0,0,1,1
GDP per capita,0,0,0,1,1
